In [ ]:
import ROOT
import math

# print(f"ROOT version: {ROOT.__version__}")
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [ ]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [ ]:
# Load experimental data for all energies from ATLAS
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')

# Function to process data for each energy block
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [ ]:
#ranges for each energy 
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)

# Extract values by energy
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# defining parameters/constants
b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25


#ensemble parameters
param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

In [ ]:
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = math.log((q2 + rho_mg_2) / lambda2) / math.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def G_p(q, a1, a2):
    q2 = q ** 2
    t = -q2
    return math.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))

def alpha_D(q2, m2_type):
    return 1.0 / (b_0 * (q2 + m2_type) * math.log((q2 + 4 * m2_type) / (lambda_qcd ** 2)))

def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


def born_amp(diff_T, s, eps, t):
    
    alpha_pomeron = 1.0 + eps + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T